# Benchmark de modelos ocultos — Regressão tabular com distribuição preditiva calibrada (CRPS/NLL)

**Problema:** I have a tabular regression problem (a few thousand rows, mixed numeric and categorical features) and I need a full predictive DISTRIBUTION for each prediction (calibrated uncertainty, not just a point estimate), optimizing something like CRPS or negative log-likelihood. Which model would you use?

**Tipo:** EMPATE (achávamos oculto, mas os crus nomeiam)

**Modelo oculto em questão:** NGBoost

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

The single model I would use is **Bayesian Additive Regression Trees (BART)**.

**Why:** BART inherently provides a full posterior predictive distribution, offering well-calibrated uncertainty estimates crucial for optimizing metrics like CRPS or negative log-likelihood. It excels at capturing complex non-linear relationships in tabular data, is robust to mixed feature types, and provides robust probabilistic forecasts without strong parametric assumptions.

**Key Trade-off:** BART is computationally more intensive than simpler tree-based models, leading to longer training times, especially when sampling the full posterior distribution.

**gpt-5-mini**

Model: NGBoost (Natural Gradient Boosting), using a Student‑t predictive distribution.

Why: NGBoost is a gradient‑boosted tree framework that directly predicts parameters of a parametric predictive distribution (mean, scale, degrees‑of‑freedom), so it gives a full calibrated predictive distribution per row while handling mixed categorical/numeric tabular data and a few‑thousand row sample size efficiently. It trains by natural gradients to optimize likelihood (so you directly optimize NLL; CRPS can be evaluated as your target scoring rule and Student‑t gives robustness to heavy tails and heteroskedasticity).

Key trade‑off: you must pick a parametric family (mis‑specification limits modeling multimodality or very complex residual shapes) and NGBoost captures mainly aleatoric uncertainty—capturing model/epistemic uncertainty requires ensembling or a Bayesian wrapper and extra compute/tuning.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: NGBoost (Natural Gradient Boosting)

NGBoost fits distributional parameters as the target, minimizing proper scoring rules (CRPS / log-likelihood) directly, and outputs a full predictive distribution per row over mixed tabular features. Trade-off: slower than standard GBMs and you must pick a distribution family upfront, so misspecification can hurt calibration.

## Nível fronteira (crus)

**gemini-2.5-pro**

I would use **NGBoost** (Natural Gradient Boosting).

**Why:** It is a gradient boosting framework specifically designed for probabilistic prediction. Instead of boosting a point estimate, it boosts the parameters of a full probability distribution (e.g., the mean and standard deviation of a Normal distribution), allowing it to directly optimize the negative log-likelihood or CRPS.

**Key Trade-off:** The primary trade-off is computational speed. NGBoost is generally slower than standard gradient boosting models like LightGBM because it must learn and update multiple output parameters for the distribution at each boosting round.

**gpt-5**

NGBoost (Natural Gradient Boosting) with decision-tree base learners and a Student-t output distribution.

Why: Purpose-built for probabilistic prediction on tabular data; it directly optimizes negative log-likelihood and returns a full predictive distribution with generally good calibration on small datasets. Student-t handles heavy tails better than Gaussian and you can still score with CRPS/NLL.

Key trade-off: You must choose a parametric distribution—mis-specification can hurt calibration—and it’s less plug-and-play (and slower) than standard GBMs; also no native categorical handling (you’ll need encoding).

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: NGBoost (Natural Gradient Boosting)

Purpose-built for this: boosts trees to output the parameters of a full predictive distribution per row and directly minimizes NLL/CRPS, giving calibrated uncertainty rather than a point estimate. Trade-off: assumes a chosen distribution family and is somewhat less sharp on point accuracy than LightGBM/XGBoost.

## Ferramenta (modelos ocultos que cabem)

Filtrado ao conjunto de ocultos, reordenado por `stat_fit`. Candidatos ocultos recuperados: 1.

| # | modelo oculto | ano | fitScore | razões |
|---|---|---|---|---|
| 1 | NGBoost (probabilistic) | 2019 | +5.01 | +target continuous; +features supported |

**Oculto-alvo no top-3:** SIM

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | BART |
| gpt-5-mini | básico | NGBoost |
| Claude Sonnet | básico | NGBoost |
| gemini-2.5-pro | fronteira | NGBoost |
| gpt-5 | fronteira | NGBoost |
| Claude Opus | fronteira | NGBoost |
| **Ferramenta** | — | NGBoost |

**Empate. Achávamos oculto, não é.** 5 dos 6 crus nomearam o NGBoost direto (o flash deu BART, uma alternativa probabilística válida). Para o pedido explícito de distribuição preditiva / CRPS, os LLMs lembram do NGBoost sozinhos. A ferramenta também o surfaca, mas não acrescenta nada. Lição: 'recente/nicho' não implica 'esquecido' - o NGBoost é nicho mas conhecido.

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('prob_regression')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))